# LC 84 — Largest Rectangle in Histogram
**Day 54 | Mixed Review Sprint | Difficulty: Hard**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Use a monotonic increasing stack
of indices. When a bar shorter than the top is found,
pop and compute rectangle: height = popped bar, width =
current index (if stack empty) else current - new top -
1. Append a sentinel 0 to flush the stack at the end.
</div>

## Official Problem Statement

Given an array of integers `heights` representing the
histogram's bar height where the width of each bar is 1,
return the area of the largest rectangle in the
histogram.

**Constraints:**
- `1 <= heights.length <= 10^5`
- `0 <= heights[i] <= 10^4`

## What This Is Actually Asking

Each bar can be the shortest bar (height limiter) of
some maximal rectangle. For each bar, we want to know
how far left and right we can extend while all bars
remain at least as tall. A monotonic stack lets us find
both boundaries in amortised O(1) per bar by exploiting
the fact that shorter bars to the right collapse the
stack. The sentinel zero at the end ensures all
remaining stack entries get processed.

## Walk Through an Example by Hand

```
heights = [2, 1, 5, 6, 2, 3]  + sentinel 0
index  =   0  1  2  3  4  5    6

stack=[]  max_area=0

i=0 h=2: stack empty, push 0          stack=[0]
i=1 h=1: 1 < heights[0]=2, POP 0
   h=2, stack empty -> width=1
   area=2*1=2  max=2
   1 >= nothing, push 1               stack=[1]
i=2 h=5: 5>1, push 2                  stack=[1,2]
i=3 h=6: 6>5, push 3                  stack=[1,2,3]
i=4 h=2: 2<6, POP 3
   h=6, new_top=2, w=4-2-1=1
   area=6*1=6  max=6
   2<5, POP 2
   h=5, new_top=1, w=4-1-1=2
   area=5*2=10  max=10
   2>=1, push 4                       stack=[1,4]
i=5 h=3: 3>2, push 5                  stack=[1,4,5]
i=6 h=0: 0<3, POP 5
   h=3, new_top=4, w=6-4-1=1
   area=3  max=10
   0<2, POP 4
   h=2, new_top=1, w=6-1-1=4
   area=2*4=8  max=10
   0<1, POP 1
   h=1, stack empty, w=6
   area=1*6=6  max=10
   stack empty, stop.

Answer: 10
```

## The Picture

```
heights = [2, 1, 5, 6, 2, 3]

      6
    5 6
    5 6
2   5 6 2 3
2 1 5 6 2 3
0 1 2 3 4 5

Largest rectangle spans indices 2-3, height=5:

    [=====]
    [=====]
    [=====]
    [=====]
    [=====]
     2   3      width=2, height=5  area=10

Monotonic stack invariant:
  stack always holds indices in increasing height order.
  When we see a shorter bar, every taller bar on the
  stack is now "blocked" on the right — compute its
  rectangle NOW.

  Pop index p:
    height = heights[p]
    if stack empty:  width = i          (extends to left edge)
    else:            width = i - stack[-1] - 1
    area = height * width
```

## When To Use This Pattern

- When finding max/min over a range that shrinks as you
  scan, think **monotonic stack**.
- When each element's contribution depends on the next
  smaller/larger element, think **stack to track
  pending computations**.
- When you need "how far can this bar extend?", think
  **stack to store left boundary, current index is
  right boundary**.
- When iterating and needing to flush remaining items
  at the end, think **sentinel value**.
- Problems like Trapping Rain Water and Daily
  Temperatures share this same stack pattern.

## The Approach

Append 0 to `heights` as a sentinel. Maintain a stack
of bar indices in non-decreasing height order. For each
bar `i`: while the stack is non-empty and `heights[i]`
is less than `heights[stack[-1]]`, pop the top index,
use its height as the rectangle height, and compute
width as `i` (if stack is now empty) or `i - stack[-1]
- 1`. Track the maximum area seen. Push each index
after processing.

In [1]:
from typing import List
from collections import defaultdict, deque

In [2]:
def test_harness(func):
    cases = [
        # (heights, expected)
        ([2, 1, 5, 6, 2, 3], 10),
        ([2, 4],              4),
        ([1],                 1),
        ([0],                 0),
        ([6, 7, 5, 2, 4, 5, 9, 3], 16),
        ([1, 1, 1, 1, 1],     5),
    ]
    passed = 0
    for heights, expected in cases:
        result = func(heights)
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        else:
            print(
                f"{status} | heights={heights} |"
                f" got={result} expected={expected}"
            )
    print(f"\nResults: {passed}/{len(cases)} passed")
    if passed == len(cases):
        print("All tests PASSED!")

In [8]:
def largestRectangleArea(heights: List[int]) -> int:
    """
    Return area of largest rectangle in histogram.

    Strategy: monotonic increasing stack of indices.
      - Append sentinel 0 to flush stack at end.
      - Pop when heights[i] < heights[stack[-1]]:
          h = heights[popped]
          w = i if stack empty else i - stack[-1] - 1
          area = h * w

    Args:
        heights: list of bar heights

    Returns:
        int: maximum rectangle area
    """


    heights = heights + [0]  # sentinel
    stack = []               # indices, increasing heights
    max_area = 0

    for i, h in enumerate(heights):
        while stack and heights[stack[-1]] > h:
            popped = stack.pop()
            ph = heights[popped]
            w = i if not stack else i - stack[-1] - 1
            area = ph * w
            max_area = max(max_area, area)
        stack.append(i)
    return max_area

print(largestRectangleArea([2,1,5,6,2,3]))  # 10
print(largestRectangleArea([2,4]))           # 4
print(largestRectangleArea([1]))             # 1
print(largestRectangleArea([6,6,6,6]))       # 24
print(largestRectangleArea([1,2,3,4,5]))     # 9
test_harness(largestRectangleArea)

10
4
1
24
9

Results: 6/6 passed
All tests PASSED!


In [ ]:
# Uncomment and run when solution is ready
# test_harness(largest_rectangle_area)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute Force (all pairs) | O(n²) | O(1) | Check every l,r pair |
| Divide & Conquer | O(n log n) | O(log n) | Elegant but harder |
| Optimal (monotonic stack) | O(n) | O(n) | Each index pushed/popped once |

## Real World Connection

At **Citi**, risk systems compute the maximum sustained
exposure window across a time series — bars are exposure
levels, and the rectangle is total risk over time.
**AWS CloudWatch** dashboard rendering uses similar
area-under-histogram calculations for capacity
planning visualisations. In **data engineering**,
this pattern appears in bin-packing optimisation:
finding the widest contiguous batch of records that
fits within a memory threshold (the height). Monotonic
stack thinking is a key skill for stream processing
and sliding-window analytics.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra